In [ ]:
import ee, eemont
import geemap
import geemap.colormaps as geecm

In [ ]:
ee.Authenticate()
ee.Initialize()

In [ ]:
from pygee import gee

# 1. Load GEE datasets + parameters

## 1.1 Parameters

Indices parameters

In [ ]:
day_beg = ee.Date('2017-07-15')
day_end = ee.Date('2023-10-15')

In [ ]:
cloud_thresh = 60

Scale and projection

In [ ]:
scale = 20
projection = "EPSG:3035"

shapefile

In [ ]:
fc_lia = ee.FeatureCollection('users/aguerou/ice_and_life/carto_h1b/lia_shp/LIA_Alps_Reinthaler_Paul_2024_Mourey_2025_with_buffer200')
geom_lia = ee.Geometry.convexHull(fc_lia)

# 2. Get S2 indices

In [ ]:
s2_indices = (ee.ImageCollection('COPERNICUS/S2_SR')
                .filter(ee.Filter.calendarRange(day_beg.getRelative("day", "year"),
                                                day_end.getRelative("day", "year"),
                                               'day_of_year'))
                .filter(ee.Filter.calendarRange(2017, 2023, 'year'))
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', cloud_thresh)) 
                .filterBounds(geom_lia)
                .scaleAndOffset()
                .maskClouds(scaledImage=True)
                .spectralIndices(["NDWI"])
            ).select(["NDWI"]).median()

In [ ]:
geemap.ee_export_image_to_asset(s2_indices.clip(fc_lia), 
                                assetId="users/aguerou/ice_and_life/carto_h1b/indices/s2_indices_gbr_ndwi_median_lia_Reinthaler_Mourey_with_buffer200", 
                                scale=scale,
                                crs=projection, 
                                maxPixels=3e9,
                                region=geom_lia)

# 3. Classification

## 3.1 Open assets indices

In [ ]:
s2_indices = ee.Image("users/aguerou/ice_and_life/carto_h1b/indices/s2_indices_gbr_ndwi_median_lia_Reinthaler_Mourey_with_buffer200")

## 3.2 Parameters

In [ ]:
water_val = 5

In [ ]:
water_thresh = [0.1, 0.15, 0.2]
slope_thresh = [90, 30, 20, 10]

## 3.3 Export water map

In [ ]:
dem = ee.Image("users/aguerou/ice_and_life/carto_h1b/data_ancillary/dem_glo30_alt_slope_aspect_alps").select("slope")

In [ ]:
for wt in water_thresh:
    for st in slope_thresh: 
        print(wt, st)
        
        water_mask = s2_indices.select("NDWI").gte(wt)
        water_classif = ee.Image.constant(0).toInt().rename("water").where(water_mask==1, water_val)

        # Filter by slope
        water_classif = water_classif.addBands(dem)
        water_classif_filter = water_classif.where(water_classif.select("slope")>st, 0).select("water")        
        
        # Export
        wt_text = str(wt).replace(".", "")
        geemap.ee_export_image_to_asset(water_classif_filter.clip(fc_lia), 
                                        assetId=f"users/aguerou/ice_and_life/carto_h1b/carto_sensitivity/water_lia_Reinthaler_Mourey_2025_withbuffer200_ndwi{wt_text}_slope{st}", 
                                        scale=scale,
                                        crs=projection, 
                                        maxPixels=3e9,
                                        region=geom_lia)

In [ ]:
Map = geemap.Map(basemap='Esri.WorldImagery')
Map.addLayer(fc_lia)
Map.addLayer(dem, {"min":0, "max":90,"palette": geecm.palettes.coolwarm})
Map.addLayer(water_classif.updateMask(water_classif.gt(0)), {"min":0, "max":5,})
Map.addLayer(water_classif_filter.updateMask(water_classif_filter.gt(0)), {"min":0, "max":5,})
Map.addLayer(ee.FeatureCollection("users/aguerou/ice_and_life/carto_h1b/lia_shp/c3s_gi_rgi11_s2_2015_v2_within_lia_reinthaler24_mourey25"))
Map